# Building a Dashboard

Combine three ECB series — the deposit facility rate, HICP inflation, and the US dollar exchange rate — into `TimeArray`s and plot them. Using **TimeSeries.jl** means no DataFrames dependency: each series is time-indexed and `merge` aligns series that share a frequency.

In [ ]:
Pollis.packages("HTTP", "CSV", "Tables", "TimeSeries", "Plots")

In [ ]:
using HTTP, CSV, Tables, TimeSeries, Dates

base = "https://data-api.ecb.europa.eu/service/data"

# Parse an ECB period: "2015-01" (monthly) or "2015-01-04" (daily)
todate(s) = length(s) == 7 ? Date(s, dateformat"yyyy-mm") : Date(s, dateformat"yyyy-mm-dd")

# Fetch one ECB series straight into a TimeArray
function ecb(flow, key, name; params = "format=csvdata")
    tbl   = CSV.File(HTTP.get("$base/$flow/$key?$params").body)
    dates = todate.(string.(Tables.getcolumn(tbl, :TIME_PERIOD)))
    vals  = Tables.getcolumn(tbl, :OBS_VALUE)
    return TimeArray(dates, vals, [name])
end

## Download the Three Series

The deposit facility rate and the exchange rate are daily (business days); HICP is monthly.

In [ ]:
p = "startPeriod=2015-01-01&format=csvdata"

dfr  = ecb("FM",  "B.U2.EUR.4F.KR.DFR.LEV", :DepositRate; params = p)
hicp = ecb("ICP", "M.U2.N.000000.4.ANR",    :HICP;        params = p)
usd  = ecb("EXR", "D.USD.EUR.SP00.A",       :USD;         params = p)

length(dfr), length(hicp), length(usd)

## Merge Series That Share a Frequency

`dfr` and `usd` are both daily, so `merge` aligns them on date (an inner join on the timestamp). Read the parts of a `TimeArray` with `timestamp`, `values`, and `colnames`.

In [ ]:
daily = merge(dfr, usd)        # inner join on date
println(colnames(daily))
last(daily, 3)

## Plot

HICP is monthly, so we keep it on its own series and let each `TimeArray` carry its own timestamps. TimeSeries provides Plots recipes, so `plot(ta)` works directly.

In [ ]:
using Plots

p1 = plot(dfr;  label = "Deposit facility rate")
plot!(p1, hicp; label = "HICP inflation (YoY)")
ylabel!(p1, "Percent")

p2 = plot(usd; label = "USD per EUR", color = :green)
ylabel!(p2, "USD/EUR")

plot(p1, p2, layout = (2, 1), size = (760, 520))

## Summary
- A small `ecb(flow, key, name)` helper returns each series as a `TimeArray` — no DataFrames needed.
- `timestamp`, `values`, and `colnames` read the parts of a TimeArray; `merge` aligns series that share a frequency.
- TimeSeries has Plots recipes, so `plot(ta)` charts a series directly.